<a href="https://colab.research.google.com/github/TheGeneral-Goat/python-file-io/blob/main/Python_database_fileIO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from datetime import datetime

# used to map indices to fields for easier recognition instead of if/else loops
index_to_field = {
    0: 'Firstname',
    1: 'Lastname',
    2: 'DOB',
    3: 'Home',
    4: 'Work',
    5: 'Mobile',
    6: 'IG'
}

# flag to maintain menu without 'while true:'
menu = True
record = []

# validates whether date of birth (DOB) is following DD-MM-YYYY format
# uses datetime.strptime to weed out invalid dates (eg 31-02)
# uses try/except to remove text
def date_validation(DOB):
    try:
        datetime.strptime(DOB, "%d-%m-%Y")
        return True
    except ValueError:
        return False


# allows users to input new record
def input_data(record):
    new_record = [""] * 7
    pointer = 0

    while pointer < len(index_to_field):
        value = input(f"Enter {index_to_field[pointer]}: ")
        if validate(value, pointer) == None:
            new_record[pointer] = value
            pointer += 1
        else:
            print("Enter a new value again.")

    record.append(new_record)
    print("Successfully inputted data.")
    return record


def validate(field, i):
    # check for blank fields
    if field.strip() == "":
        print("Blank required field.")
        return i # returns i into invalid_field (see later)

    # check if phone number is correct, length check and format check
    elif i in [3, 4, 5]:
        if len(field) > 8 or not field.isdigit():
            print(f"Invalid {index_to_field[i]} phone number")
            return i

    # format check for DOB
    elif i == 2:
        if not date_validation(field):
            print("Invalid date of birth.")
            return i

    # length check for names
    elif i in [0, 1, 6]:
        if len(field) > 25:
            print(f"Invalid {index_to_field[i]}.")
            return i

    return None

# saves data back into the text file for easy downloading
# akin to saving RAM data into SSD before closing program
def save_data():
    f = open("/data.txt", "w")
    i = 0

    while i < len(record):
        joined_record = "|||".join(record[i]) + "\n"
        f.write(joined_record)
        i += 1

    print("Data successfully saved. Goodbye.")
    f.close()


if os.path.exists("/data.txt"):
    textfile = open("/data.txt", "r")
    i = 0

    print("data.txt exists. Now opening.")
    # read textfile into record
    record = textfile.readlines()

    while i < len(record):
        invalid_fields = [] # stores invalid fields for later editing

        # replaces new lines and field seperators with nothing
        # so we can get only the data in the field
        if type(record[i]) == str:
            record[i] = record[i].replace("\n", "").strip()
            record[i] = record[i].split("|||")

        # ensures that the record[i] has the correct amount of fields (7)
        # such that program doesnt break
        if len(record[i]) != len(index_to_field):
            print(f"Record {i} has an incorrect number of fields.")
            record.pop(i)
            continue

        for j in range(len(index_to_field)):
            error_check = validate(record[i][j], j)
            if error_check is not None:
                invalid_fields.append(j)

        if len(invalid_fields) == 0:
            print(f"Record[{i}] is all correct.")
            i += 1
        # forces user to reinput data to fit validation rules based on invalid_fields
        else:
            print(f"Record {i} has at least one error. Look at the error code above for more.")
            k = 0
            while k < len(invalid_fields):
                corr = invalid_fields[k]
                new_value = input(f"Input corrected data for {index_to_field[corr]}: ")
                if validate(new_value, corr) == None:
                    record[i][corr] = new_value
                    k += 1
                else:
                    print("Correction invalid. Please try again.")
            print(f"-> Record {i} successfully corrected and updated in memory.\n")
            i += 1

    textfile.close()

else:
    print("File does not exist.")

while menu:
    choice = input("Are you reading data or inputting data? Please type 'Input' or 'Read'."
    "Press Enter to exit and save your data. "
    ).lower()

    if "input" in choice:
        input_data(record)

    elif "read" in choice:
        choice_of_field = input("Which field would you like to see? Avaliable fields are Firstname, Lastname, Home, Work, Mobile: ").lower()
        p = 0
        read_index = None
        while p < len(index_to_field) and read_index == None:
            if index_to_field[p].lower() == choice_of_field:
                read_index = p
            p += 1
        if read_index is not None:
            for count in range(len(record)):
                print(record[count][read_index])
        else:
            print("Invalid field selection. Returning to menu.")

    # print record for debugging, not seen by user
    elif choice == "record":
        print(record)

    else:
        menu = False
        save_data()